# Tools and harness: the agent is the retriever

Reference implementation 5 (RI #5) gives an agent tools over a corpus and no fixed retrieval stage. This notebook shows the tools on one question by hand, defines the arms as configurations of one tool server, steps through one reference trajectory, and puts the arms in one table with RI #4's retrievers on the shared 300 MuSiQue questions (150 two-hop, 75 three-hop, 75 four-hop), with the model held fixed.

Two results come first. The surprise is the baseline: `C0`, the same agent with only a top-5 dense search, scores F1 0.505 on all 300, above RI #4's `hipporag2` at 0.390, so most of what an agent gains here comes from the loop before grep is involved. And the existence proof does not separate the arms on this corpus: on the 75 four-hop questions `C5` reaches F1 0.457 against 0.434 for `C0`, a paired difference of +0.022 (95% bootstrap interval -0.054 to +0.104), though it sees every supporting passage of 29% of them, where RI #4's `dense` sees none in its top 10. A second run of both arms on these 75 gave +0.086, so a difference this size is within one run's noise. Over all 300, `C5` adds 0.045 F1 over `C0` (interval +0.006 to +0.086) and spends 12.8 calls and 47,600 tokens per question against `C0`'s 8.7 calls and 78,000 tokens. Both sit well under the standard of 45 calls and 590,000 tokens. Grep's lead does appear once the corpus is twelve times larger; notebook 03 has that ladder.

One arm loses. `grep-iterative` scores F1 0.240 against `dense-iterative`'s 0.297: in RI #4's scripted decomposition loop, a grep for each step's rarest capitalized span finds the right passage less often than dense retrieval does. That negative is scoped to this span rule with `Qwen3_6-35B-A3B` on MuSiQue's 11,656 passages. For it to win, the pattern would have to come from the model, as it does in `C5`; the cheapest test is one extra call per step that asks the model for the grep pattern.


The reference trajectories ran on the on-prem `Qwen3_6-35B-A3B`, the model of RI #4's rows, with thinking off and a budget of 30 model calls per question. With `RUN_EXPENSIVE = False` the notebook reads them from the cache. With it on, the last section reruns `C0` and `C5` on your model for 30 questions on the cheap tier, or 10 on a mid-tier or frontier model, and judges the answers with the shared judge.

MuSiQue (Trivedi, Balasubramanian, Khot, and Sabharwal, TACL 2022) is licensed CC BY 4.0. The cells below print a few of its questions and passages.

The table above is the on-prem reference. A later section reruns the three RI #5 arms and RI #4's `dense`, `dense-iterative`, and `hipporag2` on `gemini-3.5-flash-lite`, the default model. `C5` and `C0` tie on four-hop questions there too, and the RI #4 pipelines answer worse on Gemini while the agents answer better.

In [1]:
import json
import os
from pathlib import Path

from agent_arms import GREP_ITERATIVE, dense_search, judge_runs, live_questions, musique_corpus, run_arm, tool_server
from agent_report import compare_models, format_comparison, format_table
from aieng.advanced_rag import spend
from aieng.advanced_rag.agents.corpus_tools import STANDARD_ARMS
from aieng.advanced_rag.corpora.musique import load_musique, stratified_test_set
from aieng.advanced_rag.models import choose_models
from run_agent_arms import report


os.environ.setdefault("HF_HUB_OFFLINE", "1")

# The live cells call your model. False reads the reference trajectories.
RUN_EXPENSIVE = False
MODELS = choose_models(show=True)

_, all_questions = load_musique()
QUESTIONS = stratified_test_set(all_questions)
corpus = musique_corpus(11_656)
by_id = {q.question_id: q for q in QUESTIONS}
print(f"{len(corpus):,} passages, {corpus.characters:,} characters; {len(QUESTIONS)} questions")

family google
  generation  gemini-3.5-flash-lite   $0.30 in / $2.50 out per 1M tokens
  judging     gemini-3.8-flash        $0.75 in / $3.75 out per 1M tokens; introductory price
  extraction  gemini-3.5-flash-lite   $0.30 in / $2.50 out per 1M tokens
  embeddings  BAAI/bge-small-en-v1.5  local
key budget not set on the key; assuming $300.00 (set ADVANCED_RAG_KEY_BUDGET_USD to override), spent $135.44, left $164.56


11,656 passages, 5,869,862 characters; 300 questions


## The tools on one question

`C5` has two tools. `grep` matches a regular expression over every passage's title and text, case-insensitively, and returns the matching passages in corpus order with a preview around the first match and a count of all matches. `open` returns a passage's numbered lines in full. The question below takes four hops; each hop's answer is an entity named in the next passage, which is what lets a literal search follow the chain.

In [2]:
question = by_id["4hop1__749363_49925_13759_736921"]
print(question.question)
print("gold answer:", question.answer)
print("supporting passages:", ", ".join(question.supporting_ids))
for step in question.decomposition:
    print(f"  {step.question}  ->  {step.answer}")

In what state is the district where the man who wanted to reform the religion practiced by Innocenzo Ferrieri preached a sermon on Marian devotion?
gold answer: Saxony-Anhalt
supporting passages: musique::01864, musique::01099, musique::01088, musique::01100
  Innocenzo Ferrieri >> religion  ->  Catholic Church
  who wanted #1 to reform and address  ->  Martin Luther
  Where did #2 preach a sermon on Marian devotion a month before his death?  ->  Wittenberg
  #3 >> located in the administrative territorial entity  ->  Saxony-Anhalt


In [3]:
server = tool_server(corpus)
session = server.session(STANDARD_ARMS["C5"])
# The first hop's answer is the bridge: it is named in the second supporting passage.
bridge = question.decomposition[0].answer
found = session.call("grep", {"pattern": bridge})
print(f"grep {bridge!r}: {found['total_hits']} passages")
for match in found["matches"][:5]:
    mark = "gold" if match["doc_id"] in question.supporting_ids else "    "
    print(f"  {mark} {match['doc_id']}  {match['title'][:40]:<40}  {match['preview'][:70]}")

grep 'Catholic Church': 56 passages
       musique::00026  Szlachta                                  ...s, however, soon adopted the Reformed faiths. After the Counter-Ref
       musique::00277  Roman Catholic Archdiocese of Brisbane    ...al - Catholics (as of 2012) 2,849,000 663,000 (23.3%) Parishes 103 
       musique::00282  Évry Cathedral                            Évry Cathedral Évry Cathedral (; "Évry Cathedral of the Resurrection")
       musique::00283  Cathedral of Chihuahua                    ...oss, Our Lady of Regla, and St Francis of Assisi is the main eccles
       musique::01085  Mary, mother of Jesus                     ... practices are at times rejected, e.g., theologian Karl Barth wrote


In [4]:
opened = session.call("open", {"doc_id": question.supporting_ids[1]})
print(opened["text"][:600])
print()
print("seen:", session.trace.seen_ids[:6], "read:", session.trace.read_ids)

1: Reformation
2: Although there had been significant earlier attempts to reform the Catholic Church before Luther -- such as those of Jan Hus, Peter Waldo, and John Wycliffe -- Martin Luther is widely acknowledged to have started the Reformation with the Ninety - five Theses. Luther began by criticising the sale of indulgences, insisting that the Pope had no authority over purgatory and that the Catholic doctrine of the merits of the saints had no foundation in the Bible. The Protestant Reformation, however, would come to incorporate doctrinal changes such as a complete reliance on Scripture 

seen: ['musique::00026', 'musique::00277', 'musique::00282', 'musique::00283', 'musique::01085', 'musique::01087'] read: ['musique::01099']


## Arms are configurations

Each arm is an `ArmConfig`: the tools the model sees, their limits, the call budget, and the guards. `C0` is the closest to a classic retriever, a top-5 dense search with no other tool. `C5` has no vector index in the loop. `grep-iterative` is not an agent: it is RI #4's `dense-iterative` pipeline (decompose the question, retrieve for each step, substitute each step's answer into the next) with a literal grep for each step's rarest span in place of dense retrieval. Its decomposition, step, and answer prompts are RI #4's, character for character.

In [5]:
for arm in (STANDARD_ARMS["C0"], STANDARD_ARMS["C5"], GREP_ITERATIVE):
    print(f"{arm.name:<15} tools={','.join(arm.tools):<12} budget={arm.max_model_calls:>2}  {arm.description}")
print()
print(
    "guards on C5:",
    {
        k: v
        for k, v in STANDARD_ARMS["C5"].to_dict().items()
        if k in ("output_cap_tokens", "narrow_above", "already_read_notice", "read_before_answer")
    },
)

C0              tools=search       budget=30  search with k fixed at 5 and 512-token previews
C5              tools=grep,open    budget=30  grep and open only
grep-iterative  tools=grep         budget= 6  RI #4's dense-iterative with grep in place of dense retrieval

guards on C5: {'output_cap_tokens': 4000, 'narrow_above': 200, 'already_read_notice': True, 'read_before_answer': True}


## The reference runs

Each arm over the 300 questions is one model-produced step. `run_arm` returns your own cached run for your model when you have one, and the reference run otherwise, and prints which, with what rebuilding it on your model would cost. A full rebuild of 300 questions is not part of the hands-on notebook time; the live cell at the end runs a subset.

In [6]:
# The reference trajectories, on the on-prem model whatever family MODELS names.
# The default model's rows are read from the committed results further down.
REFERENCE = choose_models("onprem")
runs = {}
for arm in ("C0", "C5", "grep-iterative"):
    result = run_arm(arm, QUESTIONS, corpus=corpus, label="shared-11656", models=REFERENCE, live=False)
    runs[arm] = result.value

agent-runs reference artifact, built with Qwen3_6-35B-A3B on Vector's cluster at $0
           2,902 calls, 23,057,298 prompt + 329,340 completion tokens
agent-runs reference artifact, built with Qwen3_6-35B-A3B on Vector's cluster at $0
           4,141 calls, 13,885,486 prompt + 400,288 completion tokens
agent-runs reference artifact, built with Qwen3_6-35B-A3B on Vector's cluster at $0
           1,472 calls, 633,571 prompt + 18,178 completion tokens


## One reference trajectory, step by step

The `C5` trajectory for the question above. Every tool call is in the trace with its arguments, the number of passages it matched, and the passages it named and showed in full, so you can see where the agent found each supporting passage.

In [7]:
c5_run = next(r for r in runs["C5"]["runs"] if r["question_id"] == question.question_id)
gold = set(question.supporting_ids)
for event in c5_run["trace"]:
    named = [d for d in event["returned_ids"] if d in gold]
    hits = f"{event['total_hits']} hits" if event["total_hits"] is not None else ""
    args = ", ".join(
        f"{k}={v!r}" for k, v in event["arguments"].items() if k not in ("ignore_case", "start_line", "end_line")
    )
    print(f"{event['index']:>2} {event['name']:<5} {args:<55} {hits:<10} gold named: {named}")
print()
print("answer:", repr(c5_run["answer"]), " gold:", repr(question.answer), f" EM {c5_run['em']:.0f}")
print(
    f"{c5_run['model_calls']} model calls, {c5_run['prompt_tokens'] + c5_run['completion_tokens']:,} tokens, {c5_run['seconds']:.0f} s"
)

 0 grep  pattern='Innocenzo Ferrieri'                            1 hits     gold named: ['musique::01864']
 1 grep  pattern='sermon on Marian devotion'                     0 hits     gold named: []
 2 grep  pattern='Marian devotion'                               1 hits     gold named: ['musique::01088']
 3 open  doc_id='musique::01088'                                            gold named: ['musique::01088']
 4 grep  pattern='Wittenberg'                                    4 hits     gold named: ['musique::01088', 'musique::01100']
 5 open  doc_id='musique::01100'                                            gold named: ['musique::01100']

answer: 'Saxony-Anhalt'  gold: 'Saxony-Anhalt'  EM 1
7 model calls, 15,254 tokens, 49 s


## The arm table

RI #4's rows come from `graphrag_technical_support/outputs/musique_graph_test.json`, on the same questions and the same model. All-supporting recall is the share of questions with every supporting passage in hand, in two readings. `read` means shown to the model in full: for RI #4, the top 5 passages its answer prompt receives; for RI #5, the passages opened, returned whole by `search`, or shown by `grep-iterative`. `seen` adds what a result named without showing in full: for RI #4, the top 10 ranked; for RI #5, every passage a `grep` preview named. An agent can read a bridge entity off a preview without opening the passage, which is why both are shown. Seconds exclude rate-limit waits; RI #4 did not record seconds per question.

In [8]:
table = report(runs, QUESTIONS, 11_656, "shared")
print("all 300")
print(format_table(table["joined"]["all"]))
for group in ("2hop", "3hop", "4hop"):
    print()
    print(group)
    print(format_table(table["joined"][group]))

all 300
arm             source model               n     EM     F1   read   seen calls/q  tokens/q  sec/q exhaust
---------------------------------------------------------------------------------------------------------
dense           RI #4  Qwen3_6-35B-A3B   300  0.193  0.250  0.163  0.237    1.00       789      -       -
dense-iterative RI #4  Qwen3_6-35B-A3B   300  0.220  0.297  0.197  0.233    4.53      2149      -       -
hipporag2       RI #4  Qwen3_6-35B-A3B   300  0.297  0.390  0.320  0.390    1.97      2421      -       -
C0              RI #5  Qwen3_6-35B-A3B   300  0.393  0.505  0.537  0.537    8.67     77956   76.3    0.12
C5              RI #5  Qwen3_6-35B-A3B   300  0.450  0.551  0.223  0.547   12.80     47619   41.2    0.12
grep-iterative  RI #5  Qwen3_6-35B-A3B   300  0.183  0.240  0.157  0.157    4.66      2172    7.4    0.00

2hop
arm             source model               n     EM     F1   read   seen calls/q  tokens/q  sec/q exhaust
--------------------------------

In [9]:
for name, value in table["comparisons"].items():
    print(
        f"{name}: {value['mean_difference']:+.3f} F1 over {value['questions']} questions, 95% interval [{value['low']:+.3f}, {value['high']:+.3f}]"
    )

C5_minus_C0_f1_4hop: +0.022 F1 over 75 questions, 95% interval [-0.054, +0.104]
C5_minus_C0_f1_all: +0.045 F1 over 300 questions, 95% interval [+0.006, +0.086]


## The same arms on the default model

The reference runs above used the on-prem `Qwen3_6-35B-A3B`. The same three RI #5 arms were rerun on `gemini-3.5-flash-lite`, the default model for participants, with the indexes and corpus unchanged and the answers judged by the shared judge `gemini-3.8-flash`. `outputs/musique_agent_arms_gemini.json` holds the rerun, and its `produced_by` block labels it a paid-model rerun. The judged column is the share of answers the judge marks correct, and the on-prem answers were judged by the same judge.

The ranking is the same as on Qwen. On Gemini's four-hop questions `C0` reaches F1 0.498 and `C5` 0.472 (difference -0.026, interval -0.110 to +0.052), and over all 300 they are level at 0.567. The result is one run each on one answering model and one question set. For `C5` to win, `C0`'s top-5 dense search would have to miss supporting passages that grep finds. Notebook 03 runs that test on this model at 139,416 passages, and there `C5` leads on all 75 four-hop questions by +0.115 F1 (+0.030 to +0.205).

In [10]:
OUTPUTS = Path("outputs")
by_model = {
    "on-prem": json.loads((OUTPUTS / "musique_agent_arms.json").read_text(encoding="utf-8")),
    "default": json.loads((OUTPUTS / "musique_agent_arms_gemini.json").read_text(encoding="utf-8")),
}
for label, content in by_model.items():
    print(
        label,
        "->",
        content["produced_by"]["model"],
        f"({content['produced_by']['label']}), judge",
        content["produced_by"].get("judge"),
    )
for group in ("all", "4hop"):
    print()
    print(group)
    print(format_comparison(compare_models(by_model, group)))

on-prem -> Qwen3_6-35B-A3B (reference trajectories), judge gemini-3.8-flash
default -> gemini-3.5-flash-lite (paid-model rerun), judge gemini-3.8-flash

all
arm             model                   produced_by                  EM     F1  judged  calls/q  tokens/q  exhaust
------------------------------------------------------------------------------------------------------------------
C0              Qwen3_6-35B-A3B         reference trajectories    0.393  0.505   0.573     8.67     77956     0.12
C0              gemini-3.5-flash-lite   paid-model rerun          0.443  0.567   0.593     6.51     27919     0.01
C5              Qwen3_6-35B-A3B         reference trajectories    0.450  0.551   0.600    12.80     47619     0.12
C5              gemini-3.5-flash-lite   paid-model rerun          0.453  0.567   0.573    10.50     31779     0.01
grep-iterative  Qwen3_6-35B-A3B         reference trajectories    0.183  0.240   0.247     4.66      2172     0.00
grep-iterative  gemini-3.5-flash-lite 

In [11]:
for label, content in by_model.items():
    for name, value in content["comparisons"].items():
        print(
            f"{label:<8} {name}: {value['mean_difference']:+.3f} F1 over {value['questions']} questions, "
            f"95% interval [{value['low']:+.3f}, {value['high']:+.3f}]"
        )

on-prem  C5_minus_C0_f1_4hop: +0.022 F1 over 75 questions, 95% interval [-0.054, +0.104]
on-prem  C5_minus_C0_f1_all: +0.045 F1 over 300 questions, 95% interval [+0.006, +0.086]
default  C5_minus_C0_f1_4hop: -0.026 F1 over 75 questions, 95% interval [-0.110, +0.052]
default  C5_minus_C0_f1_all: +0.000 F1 over 300 questions, 95% interval [-0.041, +0.040]


### RI #4's pipelines on the default model

RI #4's `dense`, `dense-iterative`, and `hipporag2` were rerun on `gemini-3.5-flash-lite` over the same indexes, and the shared judge graded both models' answers. The surprise is the direction: every pipeline answers worse on Gemini than on Qwen, while the agents answer better. `hipporag2` falls from F1 0.390 to 0.265 and `C0` rises from 0.505 to 0.567, so the gap between them grows from 0.115 to 0.302. The pipelines' retrieval does not get worse (`dense-iterative` has every supporting passage in its top five more often on Gemini, 0.307 against 0.197), and a follow-up that handed Gemini the exact five passages Qwen answered from puts the loss on abstention: Gemini replies "unknown" to 65% to 76% of questions against Qwen's 38% to 55%, and the answers both models give score the same within 0.03 F1. An answering prompt without the "unknown" option lifts Gemini's `hipporag2` from F1 0.236 to 0.425 on Qwen's passages, above Qwen's 0.390 with the standard prompt. RI #4's README has the table. One run each, scoped to this model, the Qwen-built indexes, and these 300 questions.

In [12]:
default = by_model["default"]
print(f"all 300 on {default['produced_by']['model']}")
print(format_table(default["joined"]["all"]))
print()
print("4hop")
print(format_table(default["joined"]["4hop"]))
RI4_OUTPUTS = Path("../graphrag_technical_support/outputs")
ri4 = {
    label: json.loads((RI4_OUTPUTS / name).read_text(encoding="utf-8"))
    for label, name in (("on-prem", "musique_graph_test.json"), ("default", "musique_graph_test_gemini.json"))
}
print()
print(f"RI #4 judged by {ri4['default']['produced_by']['judge']}")
for arm in ("dense", "dense-iterative", "hipporag2"):
    for content in ri4.values():
        entry = content["arms"][arm]
        print(
            f"{arm:<16}{content['test']['model']:<24}F1 {entry['summary']['all']['f1']:.3f}  "
            f"judged {entry['judged']['all']:.3f}  four-hop judged {entry['judged']['4hop']:.3f}"
        )

all 300 on gemini-3.5-flash-lite
arm             source model               n     EM     F1   read   seen calls/q  tokens/q  sec/q exhaust
---------------------------------------------------------------------------------------------------------
dense           RI #4  gemini-3.5-flash-lite 300  0.140  0.170  0.163  0.237    0.98       748      -       -
dense-iterative RI #4  gemini-3.5-flash-lite 300  0.190  0.245  0.307  0.393    4.54      2083      -       -
hipporag2       RI #4  gemini-3.5-flash-lite 300  0.207  0.265  0.330  0.397    1.91      2227      -       -
C0              RI #5  gemini-3.5-flash-lite 300  0.443  0.567  0.600  0.600    6.51     27919    7.2    0.01
C5              RI #5  gemini-3.5-flash-lite 300  0.453  0.567  0.367  0.697   10.50     31779    9.8    0.01
grep-iterative  RI #5  gemini-3.5-flash-lite 300  0.197  0.230  0.293  0.293    4.72      2029    3.2    0.00

4hop
arm             source model               n     EM     F1   read   seen calls/q  tokens/

## Cost per question

An agent resends its growing transcript on every call, so tokens per question grow faster than calls. The standard RI #4 used to drop Microsoft GraphRAG was 45 calls and 590,000 tokens per question. Exhaustion is the share of questions that reached the 30-call budget and were made to answer.

In [13]:
for arm, entry in table["arms"].items():
    s = entry["summary"]["all"]
    print(
        f"{arm:<15} {s['calls_per_question']:>5.1f} calls  {s['tokens_per_question']:>8,.0f} tokens  {s['seconds_per_question']:>5.1f} s  exhausted {s['exhaustion_rate']:.1%}"
    )
print(f"{'standard':<15} {45:>5.1f} calls  {590_000:>8,.0f} tokens")

C0                8.7 calls    77,956 tokens   76.3 s  exhausted 11.7%
C5               12.8 calls    47,619 tokens   41.2 s  exhausted 12.0%
grep-iterative    4.7 calls     2,172 tokens    7.4 s  exhausted 0.0%
standard         45.0 calls   590,000 tokens


## Seen in a preview, never opened

The trace separates what the agent saw from what it read. A supporting passage that appeared in a `grep` preview and was never opened is a navigation failure: the search found it and the agent moved on. This column is how you tell that from a search that never surfaced the passage.

In [14]:
for arm, entry in table["arms"].items():
    cells = [
        f"{g} seen-not-read {entry['summary'][g]['seen_not_read_rate']:.2f}, all seen {entry['summary'][g]['all_supporting_recall_seen']:.2f}"
        for g in ("2hop", "3hop", "4hop")
    ]
    print(f"{arm:<15} " + " | ".join(cells))

C0              2hop seen-not-read 0.00, all seen 0.77 | 3hop seen-not-read 0.00, all seen 0.36 | 4hop seen-not-read 0.00, all seen 0.24
C5              2hop seen-not-read 0.41, all seen 0.73 | 3hop seen-not-read 0.48, all seen 0.44 | 4hop seen-not-read 0.61, all seen 0.29
grep-iterative  2hop seen-not-read 0.00, all seen 0.27 | 3hop seen-not-read 0.00, all seen 0.08 | 4hop seen-not-read 0.00, all seen 0.01


## Run `C0` and `C5` on your model

On the cheap tier this runs 30 questions (10 per hop count); on a mid-tier or frontier model, 10. The estimate prints first. Each row names its model. Compare your two arms with each other; do not rank your model against the reference row, because a trajectory depends on the model that made it.

In [15]:
subset = live_questions(QUESTIONS, MODELS.generation.model)
reference_subset = live_questions(QUESTIONS, "Qwen3_6-35B-A3B")
print(f"{len(subset)} questions on {MODELS.generation.model}")
estimates = {
    arm: run_arm(arm, reference_subset, corpus=corpus, label="live-11656", models=MODELS, live=False)
    for arm in ("C0", "C5")
}

30 questions on gemini-3.5-flash-lite
agent-runs reference artifact, built with Qwen3_6-35B-A3B on Vector's cluster at $0
           319 calls, 2,702,945 prompt + 35,868 completion tokens
           rebuilding with gemini-3.5-flash-lite: about 319 calls, est. $0.90 (set RUN_EXPENSIVE = True to rebuild)
agent-runs reference artifact, built with Qwen3_6-35B-A3B on Vector's cluster at $0
           451 calls, 1,614,941 prompt + 46,591 completion tokens
           rebuilding with gemini-3.5-flash-lite: about 451 calls, est. $0.60 (set RUN_EXPENSIVE = True to rebuild)


In [16]:
if RUN_EXPENSIVE:
    ids = {q.question_id for q in subset}
    rows = []
    for arm in ("C0", "C5"):
        mine = run_arm(
            arm,
            subset,
            corpus=corpus,
            label=f"live-{len(subset)}",
            models=MODELS,
            live=True,
            search=lambda: dense_search(corpus),
        ).value
        ours = [r for r in runs[arm]["runs"] if r["question_id"] in ids]
        for who, value_runs, model in (("yours", mine["runs"], mine["model"]), ("reference", ours, runs[arm]["model"])):
            verdicts = judge_runs(
                value_runs, subset, corpus=corpus, answered_by=model, label=f"{arm}-{who}", models=MODELS, live=True
            ).value["verdicts"]
            n = len(value_runs)
            rows.append(
                (
                    arm,
                    who,
                    model,
                    n,
                    sum(r["f1"] for r in value_runs) / n,
                    sum(verdicts.values()) / n,
                    sum(r["model_calls"] for r in value_runs) / n,
                )
            )
    print(f"{'arm':<5}{'run':<11}{'model':<24}{'n':>3}{'F1':>7}{'judged':>8}{'calls/q':>9}")
    for arm, who, model, n, f1, judged, calls in rows:
        print(f"{arm:<5}{who:<11}{model:<24}{n:>3}{f1:>7.3f}{judged:>8.3f}{calls:>9.1f}")
    print(f"judged by {MODELS.judging.model}")
else:
    print("Set RUN_EXPENSIVE = True to run C0 and C5 on your model.")

Set RUN_EXPENSIVE = True to run C0 and C5 on your model.


In [17]:
print(spend.report())

this run, estimated   est. $0.00 over 0 models
this run, proxy usage $0.00 (team key: includes teammates' calls in the same window)
key budget not set on the key; assuming $300.00 (set ADVANCED_RAG_KEY_BUDGET_USD to override), spent $135.44, left $164.56
